# Базовые модели: naive и Prophet

Две простые отправные точки для сравнения:
- **naive**: прогноз равен последнему известному значению;
- **Prophet**: стандартная модель временных рядов от Meta, по отдельной модели на каждую территорию.

Метрики и схема проверки такие же, как в остальных ноутбуках: последние 6 целевых месяцев,
горизонты 1/3/6/12 мес, MAE в рублях (дополнительно WAPE и R²). Модель на каждом разрезе
обучается только на данных до него.

In [13]:
%load_ext autoreload
%autoreload 2
from forecasting import *
import os, logging
os.environ["CMDSTANPY_LOG_LEVEL"] = "WARNING"

for name in ["cmdstanpy", "prophet"]:
    logger = logging.getLogger(name)
    logger.setLevel(logging.CRITICAL)
    logger.addHandler(logging.NullHandler())
    logger.propagate = False

logging.getLogger("cmdstanpy").disabled = True 

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [7]:
df_general = pd.read_parquet('../data/df_general.parquet')
dates_all = pd.to_datetime(np.sort(df_general['date'].unique()))
EVAL_TARGETS = list(dates_all[-6:])

# 300 случайных территорий: Prophet на всех 2190 считается слишком долго
rng = np.random.default_rng(42)
sample_ids = rng.choice(df_general['territory_id'].unique(), 300, replace=False)
df_sample = df_general[df_general['territory_id'].isin(sample_ids)]

## Данные

24 месяца (2023–2024), 2190 территорий, около 50 тысяч наблюдений. Расходы сильно отличаются
между территориями: от 11 до 91 тысячи, среднее около 28 тысяч.

In [2]:
print(df_general['total_spending'].describe())

count    50509.000000
mean     28568.014473
std      10737.544470
min      11772.000000
25%      20815.000000
50%      25031.000000
75%      33567.000000
max      91003.000000
Name: total_spending, dtype: float64


In [3]:
df_general['territory_id'].nunique()

2190

## Naive

Последнее известное значение. Нужна как самый простой ориентир: любая модель, которая её не обходит, бесполезна.

In [4]:
print('Все 2190 территорий:')
display(run_all(df_general, naive_forecast, EVAL_TARGETS))
print('300 территорий (как для Prophet):')
display(run_all(df_general, naive_forecast, EVAL_TARGETS, eval_ids=sample_ids))

Все 2190 территорий:


,MAE,WAPE,R2
horizon,,,
1,2022.744,0.061,0.944
3,2479.204,0.074,0.908
6,3527.555,0.108,0.846
12,4287.142,0.134,0.836


300 территорий (как для Prophet):


,MAE,WAPE,R2
horizon,,,
1,1934.572,0.061,0.944
3,2352.685,0.073,0.912
6,3364.180,0.107,0.850
12,4068.752,0.133,0.837


**Результат (MAE, все территории):** 2023 / 2479 / 3528 / 4287 для горизонтов 1 / 3 / 6 / 12 мес.
Ошибка растёт с горизонтом. (Первая версия этого ноутбука считала на других целевых месяцах для
каждого горизонта, поэтому сравнивать её цифры с этими нельзя, она заменена.)

## Prophet

По отдельной модели на каждую территорию. Настройки подобраны под очень короткий ряд (12–23 точки):
- `yearly_seasonality=2`: простая годовая сезонность (мало данных для сложной);
- `changepoint_prior_scale=0.001`: тренд почти не меняет направление (иначе подгоняется под шум);
- аддитивная сезонность, недельной и дневной нет (данные месячные).

**Почему 300 территорий:** Prophet обучает тысячи моделей, на всех 2190 территориях один горизонт
считался около 40 минут. Поэтому сравнение с остальными моделями делается на одной случайной
подвыборке из 300 территорий (seed 42), и остальные модели тоже проверяются на ней.

In [8]:
res_prophet = run_all(df_sample, prophet_forecast, EVAL_TARGETS, log_transform=False)
res_prophet

,MAE,WAPE,R2
horizon,,,
1,1788.799,0.056,0.955
3,2003.953,0.064,0.932
6,2347.805,0.075,0.926
12,13730.066,0.450,-1.817


**Результат (MAE, 300 территорий):**

| | 1 мес | 3 мес | 6 мес | 12 мес |
|---|---|---|---|---|
| Prophet | 1789 | 2004 | 2348 | 13730 |
| WAPE | 5,6% | 6,4% | 7,5% | 45% |

На 12 мес Prophet промахивается (R² отрицательный, хуже, чем просто среднее): он продолжает тренд,
выученный по 12 точкам. На коротких горизонтах результат приличный, но заметно хуже seasonal naive.

### Вариант с логарифмом

Идея: логарифм делает сезонность мультипликативной (в процентах), что ближе к росту расходов.

In [14]:
res_prophet_log = run_all(df_sample, prophet_forecast, EVAL_TARGETS, log_transform=True)
res_prophet_log

,MAE,WAPE,R2
horizon,,,
1,1879.305,0.060,0.947
3,2260.059,0.072,0.723
6,2519.144,0.081,0.899
12,156215.783,5.136,-113469.200


| MAE | 1 мес | 3 мес | 6 мес | 12 мес |
|---|---|---|---|---|
| Prophet | 1789 | 2004 | 2348 | 13730 |
| Prophet + log | 1879 | 2260 | 2519 | 156216 |

Логарифм **ухудшил** результат на всех горизонтах, а на 12 мес прогноз улетел (экспонента от тренда,
продолженного по 12 точкам). В сравнении используется обычный Prophet.

## Выводы
- Даже на коротких горизонтах Prophet проигрывает seasonal naive (см. `02_forecast.ipynb`),
  на 12 мес он непригоден: с одним годом данных годовой цикл и тренд не выучить.
- Эксперимент с логарифмом не помог.